# Kill-Test Evaluation & Manual Crux Audit

**Track 4 — Crux-Audited Consensus Capstone**

This notebook runs the full kill-test evaluation pipeline:

1. **Data loading**: read all stage JSONL files, join by `id`/`crux_id`
2. **Table 1 (KC1)**: crux-route accuracy vs. full-question baseline + McNemar tests
3. **Table 2**: end-to-end oversight comparison (Base DR, stronger consultant, Crux-Audited Consensus)
4. **Plot 1 (KC2)**: calibration & AUROC for checkability signals (c_cons, c_prob, c_2judge, d×c)
5. **Table 3 (KC3)**: leakage analysis, layer comparison, balanced subsample
6. **Tau selection** on the dev split
7. **50-sample manual audit**: stratified sampling, rubric generation, agreement metrics
8. **Go/No-Go** threshold check

In [ ]:
import sys, os
from pathlib import Path

# Ensure the src directory is on the path
REPO_ROOT = Path(os.getcwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from crux.io import read_jsonl, index_by_id, REPO_ROOT as _REPO
from crux.track4.kill_test_eval import (
    CaseRecord, filter_split, balanced_subsample,
    table1_kill_criterion_1, table2_oversight,
    plot1_calibration, table3_leakage_and_layers,
    select_tau, summary_stats, mcnemar_test,
    load_case_records,
)
from crux.track4.manual_audit import (
    stratified_sample, build_audit_rows, write_audit_csv,
    per_column_agreement, check_audit_agreement,
    RUBRIC_COLUMNS, RUBRIC_DESCRIPTIONS, AUDIT_AGREEMENT_THRESHOLDS,
)

print(f"Repo root: {_REPO}")

## 1. Data Loading

Load all stage files with `with_labels=True` (Track 4 is the designated consumer of ground truth).

**Note**: Update the paths below to point to your actual stage JSONL files once the pipeline has run.

In [ ]:
# ── Configure data directory ─────────────────────────────────────
DATA_DIR = _REPO / "data"

# ── Load all stage files via the Track 4 loader ──────────────────
# load_case_records joins cases, crux_proposals, leakage, and
# decisiveness stage files into flat CaseRecord objects.
# Update DATA_DIR once the pipeline produces the real stage files.

try:
    all_cases = load_case_records(DATA_DIR)
    print(f"Loaded {len(all_cases)} CaseRecord objects via load_case_records")
except FileNotFoundError as e:
    print(f"Stage files not found ({e}); falling back to seed data only")
    # Fallback: load just the seed cases file for plumbing checks
    CASES_PATH = DATA_DIR / "seed" / "seed_30.jsonl"
    cases_raw = read_jsonl(CASES_PATH, "cases", with_labels=True)
    all_cases = [
        CaseRecord(
            id=c["id"],
            dataset=c.get("dataset", ""),
            source=c.get("source", ""),
            split=c.get("split", ""),
            ground_truth=c.get("eval", {}).get("ground_truth"),
            initially_right=c.get("eval", {}).get("initially_right"),
        )
        for c in cases_raw
    ]
    print(f"  Loaded {len(all_cases)} seed CaseRecord objects (plumbing only)")

print(f"  kill-test eligible: {sum(c.is_kill_test for c in all_cases)}")
print(f"  with ground truth:  {sum(c.ground_truth is not None for c in all_cases)}")

In [ ]:
# ── Split into dev / test ───────────────────────────────────────
dev_cases  = filter_split(all_cases, "dev")
test_cases = filter_split(all_cases, "test")

print(f"Dev:  {len(dev_cases)} cases")
print(f"Test: {len(test_cases)} cases")

# Summary statistics
import json as _json
print("\nOverall summary:")
print(_json.dumps(summary_stats(all_cases), indent=2))

## 2. Tau Selection (Dev Split Only)

Choose the d×c threshold tau on the **dev** split that maximises accuracy × coverage.
This tau is then fixed for all test-split evaluation.

In [ ]:
if dev_cases:
    tau_result = select_tau(dev_cases, c_signal="c_cons")
    print(f"Best tau: {tau_result['best_tau']}")
    print(f"  accuracy:  {tau_result['best']['accuracy']:.3f}")
    print(f"  coverage:  {tau_result['best']['coverage']:.3f}")
    print(f"  score:     {tau_result['best']['score']:.3f}")
    print("\nFull sweep:")
    for row in tau_result["sweep"]:
        print(f"  tau={row['tau']:.1f}  acc={row['accuracy']:.3f}  cov={row['coverage']:.3f}  score={row['score']:.3f}")
else:
    print("No dev-split cases available; using default tau=0.3")
    tau_result = {"best_tau": 0.3}

## 3. Table 1 — Kill Criterion 1 (End-to-End Accuracy)

Crux-route accuracy vs. full-question 2-way baseline, with McNemar p-values.

| Method | Accuracy | Coverage | McNemar p |
|--------|----------|----------|-----------|

In [ ]:
eval_cases = test_cases if test_cases else [c for c in all_cases if c.is_kill_test]
print(f"Evaluating on {len(eval_cases)} cases\n")

t1 = table1_kill_criterion_1(eval_cases)

# Display as a table
print(f"{'Method':<25} {'Accuracy':>10} {'Coverage':>10} {'McNemar p':>12} {'N answered':>12}")
print("-" * 75)
for row in t1:
    acc = f"{row['accuracy']:.3f}" if row.get('accuracy') is not None else "—"
    cov = f"{row['coverage']:.3f}" if row.get('coverage') is not None else "—"
    p = f"{row['mcnemar_p']:.4f}" if row.get('mcnemar_p') is not None else "—"
    n = str(row.get('n_answered', row.get('n_total', '')))
    print(f"{row['method']:<25} {acc:>10} {cov:>10} {p:>12} {n:>12}")

## 4. Table 2 — End-to-End Oversight Comparison

Base DR vs. stronger consultant vs. Crux-Audited Consensus:
accuracy, agreement-trap catch rate, false-overturn rate.

In [ ]:
t2 = table2_oversight(eval_cases)

print(f"{'Method':<30} {'Acc':>8} {'Trap catch':>12} {'False flip':>12} {'N':>6}")
print("-" * 72)
for row in t2:
    acc = f"{row['accuracy']:.3f}" if row.get('accuracy') is not None else "—"
    trap = f"{row['trap_catch_rate']:.3f}" if row.get('trap_catch_rate') is not None else "—"
    flip = f"{row['false_overturn_rate']:.3f}" if row.get('false_overturn_rate') is not None else "—"
    print(f"{row['method']:<30} {acc:>8} {trap:>12} {flip:>12} {row['n']:>6}")

## 5. Plot 1 — Kill Criterion 2 (Calibration & AUROC)

Calibration buckets for each checkability signal + d×c, AUROC, and count of confidently-wrong cases.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt

cal = plot1_calibration(eval_cases)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Kill Criterion 2: Calibration & AUROC", fontsize=14)

for idx, (sig_name, sig_data) in enumerate(cal.items()):
    ax = axes.flat[idx]
    buckets = sig_data["buckets"]
    if not buckets:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(sig_name)
        continue
    
    mids = [(b["bucket_lo"] + b["bucket_hi"]) / 2 for b in buckets]
    accs = [b["accuracy"] if b["accuracy"] is not None else 0 for b in buckets]
    ns   = [b["n"] for b in buckets]
    
    ax.bar(mids, accs, width=(mids[1] - mids[0]) * 0.8 if len(mids) > 1 else 0.15,
           alpha=0.7, color="steelblue", edgecolor="white")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="perfect calibration")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel(sig_name)
    ax.set_ylabel("Accuracy")
    auroc_val = sig_data["auroc"]
    auroc_str = f"{auroc_val:.3f}" if not (auroc_val != auroc_val) else "N/A"  # NaN check
    ax.set_title(f"{sig_name}\nAUROC={auroc_str}, conf-wrong={sig_data['n_confident_wrong']}")

# Hide extra subplot if odd number of signals
for idx in range(len(cal), len(axes.flat)):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig("kc2_calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: kc2_calibration.png")

In [ ]:
# AUROC summary table
print(f"{'Signal':<20} {'AUROC':>8} {'N':>6} {'Conf-wrong':>12}")
print("-" * 50)
for sig_name, sig_data in cal.items():
    auroc_val = sig_data["auroc"]
    auroc_str = f"{auroc_val:.3f}" if not (auroc_val != auroc_val) else "N/A"
    print(f"{sig_name:<20} {auroc_str:>8} {sig_data['n_total']:>6} {sig_data['n_confident_wrong']:>12}")

# Build candidate-level data for leakage analysis from CaseRecords
# (When full stage files are available, you can also load raw leakage
#  records for more granular candidate-level stats.)
all_candidates = []
for c in eval_cases:
    if c.selected_crux_id:
        all_candidates.append({
            "crux_id": c.selected_crux_id,
            "case_id": c.id,
            "is_leaky": c.is_leaky,
            "d": c.crux_d,
        })

t3 = table3_leakage_and_layers(eval_cases, all_candidates if all_candidates else None)

print("=== Leakage ===")
lk = t3["leakage"]
print(f"  Candidates:    {lk['n_candidates']}")
print(f"  Leaky:         {lk['n_leaky']}")
leaky_share = f"{lk['leaky_share']:.1%}" if lk['leaky_share'] is not None else "N/A"
print(f"  Leaky share:   {leaky_share}")
print(f"  Leaky d mean:  {lk['leaky_d_mean']}")
print(f"  Clean d mean:  {lk['nonleaky_d_mean']}")

print("\n=== KC1 with/without leaky ===")
for key in ("kc1_with_leaky", "kc1_without_leaky"):
    row = t3[key]
    acc = f"{row['accuracy']:.3f}" if row['accuracy'] is not None else "N/A"
    print(f"  {key}: n={row['n']}, accuracy={acc}")

print("\n=== Layer Comparison ===")
ly = t3["layers"]
print(f"  N cases:                {ly['n_cases']}")
cross = f"{ly['cross_family_agreement_rate']:.3f}" if ly['cross_family_agreement_rate'] is not None else "N/A"
same  = f"{ly['same_family_agreement_rate']:.3f}" if ly['same_family_agreement_rate'] is not None else "N/A"
print(f"  Cross-family agreement: {cross}")
print(f"  Same-family agreement:  {same}")
print(f"  Cross-family catches:   {ly['cross_family_catch_count']}")
print(f"  Same-family catches:    {ly['same_family_catch_count']}")

In [ ]:
# Build candidate-level data for leakage analysis
all_candidates = []
for rec in leakage_records:
    cid = rec.get("crux_id", "")
    case_id = cid.split("#")[0] if "#" in cid else cid
    d_val = None
    if cid in dec_by_id:
        d_val = dec_by_id[cid].get("d")
    all_candidates.append({
        "crux_id": cid,
        "case_id": case_id,
        "is_leaky": rec.get("is_leaky", False),
        "d": d_val,
    })

t3 = table3_leakage_and_layers(eval_cases, all_candidates if all_candidates else None)

print("=== Leakage ===")
lk = t3["leakage"]
print(f"  Candidates:    {lk['n_candidates']}")
print(f"  Leaky:         {lk['n_leaky']}")
leaky_share = f"{lk['leaky_share']:.1%}" if lk['leaky_share'] is not None else "N/A"
print(f"  Leaky share:   {leaky_share}")
print(f"  Leaky d mean:  {lk['leaky_d_mean']}")
print(f"  Clean d mean:  {lk['nonleaky_d_mean']}")

print("\n=== KC1 with/without leaky ===")
for key in ("kc1_with_leaky", "kc1_without_leaky"):
    row = t3[key]
    acc = f"{row['accuracy']:.3f}" if row['accuracy'] is not None else "N/A"
    print(f"  {key}: n={row['n']}, accuracy={acc}")

print("\n=== Layer Comparison ===")
ly = t3["layers"]
print(f"  N cases:                {ly['n_cases']}")
cross = f"{ly['cross_family_agreement_rate']:.3f}" if ly['cross_family_agreement_rate'] is not None else "N/A"
same  = f"{ly['same_family_agreement_rate']:.3f}" if ly['same_family_agreement_rate'] is not None else "N/A"
print(f"  Cross-family agreement: {cross}")
print(f"  Same-family agreement:  {same}")
print(f"  Cross-family catches:   {ly['cross_family_catch_count']}")
print(f"  Same-family catches:    {ly['same_family_catch_count']}")

## 7. Balanced Subsample Analysis

Re-run KC1 on a balanced subsample where each consultant is initially right ~50% of cases (Appendix C.2).

In [ ]:
if eval_cases:
    bal = balanced_subsample(eval_cases, seed=42)
    print(f"Balanced subsample: {len(bal)} cases")
    a_right = sum(1 for c in bal if c.initially_right == "a")
    b_right = sum(1 for c in bal if c.initially_right == "b")
    print(f"  A right: {a_right}, B right: {b_right}")
    
    t1_bal = table1_kill_criterion_1(bal)
    print(f"\n{'Method':<25} {'Accuracy':>10} {'Coverage':>10} {'N answered':>12}")
    print("-" * 60)
    for row in t1_bal:
        acc = f"{row['accuracy']:.3f}" if row.get('accuracy') is not None else "—"
        cov = f"{row['coverage']:.3f}" if row.get('coverage') is not None else "—"
        n = str(row.get('n_answered', row.get('n_total', '')))
        print(f"{row['method']:<25} {acc:>10} {cov:>10} {n:>12}")
else:
    print("No evaluation cases available for balanced subsample analysis.")

In [ ]:
# Draw the stratified sample for manual audit
audit_pool = [c.__dict__ for c in eval_cases]  # convert to dicts for sampling
sampled = stratified_sample(audit_pool, n=50, seed=42)
print(f"Sampled {len(sampled)} cases for manual audit")

# Show dataset distribution
from collections import Counter
dist = Counter(c.get("dataset", "unknown") for c in sampled)
for k, v in sorted(dist.items()):
    print(f"  {k}: {v}")

In [ ]:
# Generate the audit CSV
audit_rows = build_audit_rows(sampled)
audit_csv_path = "manual_audit_sheet.csv"
write_audit_csv(audit_rows, audit_csv_path)
print(f"Wrote {len(audit_rows)} rows to {audit_csv_path}")
print(f"\nRubric columns: {len(RUBRIC_COLUMNS)}")
print("\nHuman annotation rubric:")
for col, desc in RUBRIC_DESCRIPTIONS.items():
    print(f"  {col}: {desc}")

In [ ]:
# ── Inter-rater agreement (run after two annotators fill the sheet) ──

# Placeholder: load two annotators' completed sheets
# annotator_a = csv.DictReader(open("audit_annotator_a.csv"))
# annotator_b = csv.DictReader(open("audit_annotator_b.csv"))
# agreement = per_column_agreement(list(annotator_a), list(annotator_b))

# Example with synthetic data for demonstration
import random
random.seed(42)
n_demo = min(50, len(sampled))

demo_a = [{"human_crux_valid": random.randint(0,1), "human_crux_clear": random.randint(0,1),
           "human_leakage_flag": random.randint(0,1), "human_decisive": random.randint(0,1),
           "human_verdict_correct": random.randint(0,1)} for _ in range(n_demo)]
demo_b = [{"human_crux_valid": random.randint(0,1), "human_crux_clear": random.randint(0,1),
           "human_leakage_flag": random.randint(0,1), "human_decisive": random.randint(0,1),
           "human_verdict_correct": random.randint(0,1)} for _ in range(n_demo)]

agreement = per_column_agreement(demo_a, demo_b)

print(f"{'Column':<25} {'% Agree':>10} {'Kappa':>8} {'N':>6}")
print("-" * 52)
for col, metrics in agreement.items():
    print(f"{col:<25} {metrics['percent_agreement']:>10.3f} {metrics['cohens_kappa']:>8.3f} {metrics['n']:>6}")

## 9. Go/No-Go Threshold Check

Check whether the inter-rater agreement meets the pre-registered thresholds.

In [ ]:
gng = check_audit_agreement(agreement)

print(f"Overall verdict: {gng['verdict']}\n")
print(f"{'Column':<25} {'Kappa':>8} {'Thresh':>8} {'Pass?':>7} {'% Agree':>10} {'Thresh':>8} {'Pass?':>7}")
print("-" * 80)
for col, result in gng["columns"].items():
    k_pass = "✓" if result["kappa_pass"] else "✗"
    p_pass = "✓" if result["pct_pass"] else "✗"
    print(f"{col:<25} {result['kappa']:>8.3f} {result['kappa_threshold']:>8.1f} {k_pass:>7} "
          f"{result['percent_agreement']:>10.3f} {result['pct_threshold']:>8.0%} {p_pass:>7}")

print(f"\nAudit agreement thresholds (pre-registered):")
for col, thresh in AUDIT_AGREEMENT_THRESHOLDS.items():
    print(f"  {col}: {thresh['description']}")